# Data Quality Cleaning
This notebook validates the raw loan dataset and applies only the cleaning needed to correct genuine data-quality issues.

In [1]:
import pandas as pd
from pathlib import Path

data_path = Path('../data/Loan_default.csv')
output_path = Path('../outputs/cleaned_loan_default.csv')

df = pd.read_csv(data_path)
df.head()

,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


In [2]:
missing_values = df.isna().sum()
missing_values

LoanID            0
Age               0
Income            0
LoanAmount        0
CreditScore       0
MonthsEmployed    0
NumCreditLines    0
InterestRate      0
LoanTerm          0
DTIRatio          0
Education         0
EmploymentType    0
MaritalStatus     0
HasMortgage       0
HasDependents     0
LoanPurpose       0
HasCoSigner       0
Default           0
dtype: int64

In [3]:
duplicate_rows = int(df.duplicated().sum())
duplicate_rows

0

In [4]:
valid_ranges = {
    'Age': (18, 69),
    'Income': (15000, 149999),
    'LoanAmount': (5000, 249999),
    'CreditScore': (300, 849),
    'MonthsEmployed': (0, 119),
    'NumCreditLines': (1, 4),
    'InterestRate': (2, 25),
    'LoanTerm': (12, 60),
    'DTIRatio': (0.1, 0.9),
    'Default': (0, 1),
}

invalid_counts = {}
for col, (low, high) in valid_ranges.items():
    invalid_count = int(((df[col] < low) | (df[col] > high)).sum())
    invalid_counts[col] = invalid_count

invalid_counts

{'Age': 0,
 'Income': 0,
 'LoanAmount': 0,
 'CreditScore': 0,
 'MonthsEmployed': 0,
 'NumCreditLines': 0,
 'InterestRate': 0,
 'LoanTerm': 0,
 'DTIRatio': 0,
 'Default': 0}

In [5]:
valid_categories = {
    'Education': ["Bachelor's", 'High School', "Master's", 'PhD'],
    'EmploymentType': ['Full-time', 'Part-time', 'Self-employed', 'Unemployed'],
    'MaritalStatus': ['Divorced', 'Married', 'Single'],
    'HasMortgage': ['Yes', 'No'],
    'HasDependents': ['Yes', 'No'],
    'LoanPurpose': ['Auto', 'Business', 'Education', 'Home', 'Other'],
    'HasCoSigner': ['Yes', 'No'],
}

inconsistent_categories = {}
for col, valid_values in valid_categories.items():
    actual_values = sorted(df[col].dropna().astype(str).str.strip().unique().tolist())
    invalid_values = sorted(set(actual_values) - set(valid_values))
    inconsistent_categories[col] = invalid_values

inconsistent_categories

{'Education': [],
 'EmploymentType': [],
 'MaritalStatus': [],
 'HasMortgage': [],
 'HasDependents': [],
 'LoanPurpose': [],
 'HasCoSigner': []}

In [6]:
df_clean = df.copy()

# Remove leading/trailing whitespace only where necessary for data consistency.
for col in df_clean.select_dtypes(include=['object']).columns:
    df_clean[col] = df_clean[col].astype(str).str.strip()

# Drop exact duplicate rows only if they exist.
if df_clean.duplicated().any():
    df_clean = df_clean.drop_duplicates().reset_index(drop=True)

# Handle missing values only when present; otherwise preserve the original data.
if df_clean.isna().any().any():
    df_clean = df_clean.dropna().reset_index(drop=True)

df_clean.shape

C:\Users\divanshu kumawat\AppData\Local\Temp\ipykernel_15880\3497214658.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_clean.select_dtypes(include=['object']).columns:


(255347, 18)

In [7]:
output_path.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(output_path, index=False)
print(f'Saved cleaned dataset to {output_path}')
print(df_clean.shape)

Saved cleaned dataset to ..\outputs\cleaned_loan_default.csv
(255347, 18)
